## Middleware

#### Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

#### • Tracking agent behaviour with loggin, analytics, and debugging
#### • Transforming prompts, tool selection, and output formatting.
#### • Adding retries, fallback, and early termination logic.
#### • Applying rate limiting, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summmarization Middleware

#### Automatically summarize conversation history when approaching token limits, perserving recent messages while compressing older context. Summarization is useful for the following:

#### • Long-running conversation that exceed context windows.
#### • Multi-turn dialogies with extensive history.
#### • Applications where preserving full conversation context matters.


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [4]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}


In [9]:
## Alternative test data

questions = [
    "What is 2+27?",
    "What is 5*2?",
    "What is 932/2?",
    "What is 32-3?",
    "What is 42*2?",
    "What is 93-2"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nProvide correct answers to the user's arithmetic questions.\n\n## SUMMARY\n- The user is asking a series of arithmetic problems, expecting accurate responses.\n- The AI has already answered:\n  -\u202f5\u202f×\u202f2\u202f=\u202f10\n  -\u202f932\u202f÷\u202f2\u202f=\u202f466\n  -\u202f32\u202f−\u202f3\u202f=\u202f29\n- The latest user query, “What is 42*2?”, has not yet been answered.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAnswer the pending question:\u202f42\u202f×\u202f2\u202f=\u202f84.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f2e8c5f7-f111-4b1c-8efc-b1cc3e2a2130'), AIMessage(content='\\(42 \\times 2 = 84\\).', additional_kwargs={'reasoning_content': 'User asks arithmetic: 42*2 = 84. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 257, 'total_tokens': 293, 'completion_time': 0.076451869

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

## Token Size

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: 
        1. Grand Hotel -5 star, ₹ 10000/night, spa, pool, gym
        2. City Inn  - 4 star, ₹ 6000/night, business center
        3. Budget Stay - 3 star, ₹ 1000/ night free wi-fi
    """

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable":{"thread_id": "test-1"}}

# Toke counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4     # // divide and get floor value,  4 chars ≈ 1 token



In [22]:
## Run Test 

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")



Paris: ~287 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='21dd365e-d32d-487a-af64-1328c6dada21'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to find hotels in Paris. Use the search_hotels function.', 'tool_calls': [{'id': 'fc_e2b4623d-7770-4f7e-9b61-c9275239cfd6', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 129, 'total_tokens': 173, 'completion_time': 0.092026928, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005065702, 'prompt_tokens_details': None, 'queue_time': 0.352283943, 'total_time': 0.09709263}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a19c2d-0c9b-710f-a1ae-25bf4c32d29a-0', tool_cal

### Fraction

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: 
        1. Grand Hotel -5 star, ₹ 10000/night, spa, pool, gym
        2. City Inn  - 4 star, ₹ 6000/night, business center
        3. Budget Stay - 3 star, ₹ 1000/ night free wi-fi
    """

# Low Fraction of testing!!
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("fraction",0.005),      # 0.05% = ~640 tokens  
            keep=("fraction",0.002)          # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable":{"thread_id": "test-1"}}

# Toke counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4     # // divide and get floor value,  4 chars ≈ 1 token



In [27]:
## Run Test 

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(f"{(response['messages'])}")

Paris: ~2141 tokens (1.6727%), 8 msgs
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nAssist the user in finding suitable hotels for their travel plans (Tokyo, New\u202fYork, Paris, and London) by gathering detailed preferences and then generating curated hotel lists (12‑20 per city) or performing real‑time availability checks.\n\n## SUMMARY\n- Provided placeholder hotel lists (3 hotels each) for Tokyo, New\u202fYork, Paris, and London via `search_hotels` tool calls, displayed in markdown tables with star rating, price, key amenities, and neighbourhood/landmark.  \n- Presented a checklist of preference categories (budget/price range, star rating, neighbourhood, amenities, travel dates & length of stay, number of rooms/guests, special requests, preferred currency) to refine the search for each city.  \n- User asked “Find hotels in Tokyo”; assistant re‑presented the Tokyo table and reiterated the preference‑gathering checklist.  \n- User then a

## Human In the loop Middleware

#### Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

#### • High-stakes operations requiring human approval(e.g database writes, financial transactions).
#### • Compliance workflows where human oversight is mandatory.
#### • Long-running coversations where human feedback guides the agent.

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""

    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}"



In [33]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]

)

In [34]:
config = {"configurable":{"thread_id":"test-approve"}}

# Step 1: Request

result = agent.invoke(
    {
        "messages": [HumanMessage(content="Send email to raj@test.com with subject 'Hello' and body 'How are you?'")]
    },
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="Send email to raj@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0d9213c2-c3ab-414f-94c7-bf198a1fabd9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_b9adf72f-4c6e-4e77-80ee-3ed710bfe9d2', 'function': {'arguments': '{"body":"How are you?","recipient":"raj@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.123837909, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008051063, 'prompt_tokens_details': None, 'queue_time': 0.310832607, 'total_time': 0.131888972}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

In [36]:
## Step2 : Approve

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions" :[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Your email has been sent to **raj@test.com** with the subject **“Hello”** and the body **“How are you?”**. Let me know if there’s anything else you’d like to do!


In [37]:
result

{'messages': [HumanMessage(content="Send email to raj@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0d9213c2-c3ab-414f-94c7-bf198a1fabd9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_b9adf72f-4c6e-4e77-80ee-3ed710bfe9d2', 'function': {'arguments': '{"body":"How are you?","recipient":"raj@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.123837909, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.008051063, 'prompt_tokens_details': None, 'queue_time': 0.310832607, 'total_time': 0.131888972}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

### Reject


In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""

    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]

)


In [41]:
config = {"configurable":{"thread_id":"test-reject"}}

# Step 1: Request

result = agent.invoke(
    {
        "messages": [HumanMessage(content="Send email to raj@test.com with subject 'Hello' and body 'How are you?'")]
    },
    config=config)

In [42]:
## Step2 : Reject

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions" :[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m unable to send the email right now because the request to use the email‑sending tool was rejected. If you’d still like me to send the message, just let me know and I can try again.


In [43]:
result

{'messages': [HumanMessage(content="Send email to raj@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='25efe11c-baf0-4ce7-853a-e70d7fd0f8f5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_4d98d47a-9c07-4294-81d8-7743e1fded42', 'function': {'arguments': '{"body":"How are you?","recipient":"raj@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.122201302, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.041036681, 'prompt_tokens_details': None, 'queue_time': 0.343749579, 'total_time': 0.163237983}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

### Editing

In [53]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""

    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]

)


In [54]:
config = {"configurable":{"thread_id":"test-edit"}}

# Step 1: Request (with wrong info)

result = agent.invoke(
    {
        "messages": [HumanMessage(content="Send email to raj@test.com with subject 'Test' and body 'Hello'")]
    },
    config=config)

In [55]:
result

{'messages': [HumanMessage(content="Send email to raj@test.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='161b9ccc-8953-43c4-835b-0b7dd4eff008'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_a1e50d1a-6b3c-4617-8192-44baed9faa84', 'function': {'arguments': '{"body":"Hello","recipient":"raj@test.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 172, 'total_tokens': 228, 'completion_time': 0.118896425, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.009695388, 'prompt_tokens_details': None, 'queue_time': 0.350236753, 'total_time': 0.128591813}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r

In [56]:
## Step2 : Edit and approve

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions" :[
                    {"type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",     # Tool name
                            "args":{                       # New arguments
                                "recipient":"correct@email.com",
                                "subject":"Corrected subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                    
                    }
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Your email has been sent. Let me know if there’s anything else I can help with!


In [57]:
result

{'messages': [HumanMessage(content="Send email to raj@test.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='161b9ccc-8953-43c4-835b-0b7dd4eff008'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_a1e50d1a-6b3c-4617-8192-44baed9faa84', 'function': {'arguments': '{"body":"Hello","recipient":"raj@test.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 172, 'total_tokens': 228, 'completion_time': 0.118896425, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.009695388, 'prompt_tokens_details': None, 'queue_time': 0.350236753, 'total_time': 0.128591813}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r